# AR(p) processes and forward sliding-window embeddings

This notebook simulates a stationary AR(p) process

$$X_t = c + \phi_1X_{t-1}+\cdots+\phi_pX_{t-p}+\varepsilon_t,$$

with centered, scaled Beta innovations

$$B_t\sim\operatorname{Beta}(\alpha,\beta),\qquad
\varepsilon_t=w\left(B_t-\frac{\alpha}{\alpha+\beta}\right).$$

The innovation support has total width $w$. The defaults $\alpha=\beta=1$ and $w=2$ give exactly the previous $\mathcal U[-1,1]$ model. The notebook plots the time series and the forward embeddings

$$Y_t^{(k)}=(X_t,X_{t+1},\ldots,X_{t+k-1})$$

for $k=2$ and $k=3$. The 3D Plotly figure can be rotated, zoomed, and panned.

Geometric overlays:

- **Mean-innovation hyperplane:** the AR relation at $\varepsilon_t=\mathbb E[\varepsilon_t]=0$.
- **Support boundaries:** the two border hyperplanes obtained by adding the lower and upper centered-Beta innovation bounds.

> An exact AR innovation hyperplane can be drawn only when the visible window contains all required lags, i.e. $p\leq k-1$. Thus $k=2$ supports an exact AR(1) line, while $k=3$ supports AR(1) and AR(2) planes. The notebook reports when a chosen order is too large for an exact overlay.


In [6]:
# If needed, uncomment the next line once:
# %pip install numpy matplotlib plotly ipywidgets

import ast
import sys
from pathlib import Path

import ipywidgets as widgets
import numpy as np
from IPython.display import clear_output, display

AR_CODE_DIR = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            root / "Code" / "Code_AR_processes",
            root,
        )
        if (candidate / "functions.py").is_file()
        and (candidate / "plotting_functions.py").is_file()
    ),
    None,
)
if AR_CODE_DIR is None:
    raise FileNotFoundError(
        "Could not locate Code/Code_AR_processes from the current directory."
    )
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Code.Code_AR_processes.functions import (
    centered_beta_support,
    simulate_beta_ar,
    sliding_window_point_cloud,
    companion_matrix,
    is_stationary,
)
from Code.Code_AR_processes.plotting_functions import (
    plot_time_series,
    plot_embedding_k2,
    plot_embedding_k3,
)


ImportError: cannot import name 'centered_beta_support' from 'Code.Code_AR_processes.functions' (/Users/iljasukhanov/MASTERARBEIT/Code/Code_AR_processes/functions.py)

## Simulation and geometry functions

In [3]:
def parse_phi(text):
    """Parse entries such as '0.7', '0.6, -0.2', or '[0.6, -0.2]' safely."""
    text = text.strip()
    if not text:
        raise ValueError("Please enter at least one AR coefficient.")
    if text.startswith("[") or text.startswith("("):
        values = ast.literal_eval(text)
    else:
        values = [float(part.strip()) for part in text.split(",")]
    phi = np.atleast_1d(np.asarray(values, dtype=float))
    if phi.ndim != 1 or len(phi) == 0 or not np.all(np.isfinite(phi)):
        raise ValueError("AR coefficients must be a finite one-dimensional sequence.")
    return phi

In [ ]:
def run_experiment(
    phi,
    n,
    intercept,
    support_width,
    beta_alpha,
    beta_beta,
    burnin,
    seed,
    max_points=2500,
):
    stationary, eigvals = is_stationary(phi)
    if not stationary:
        raise ValueError(
            "The selected AR coefficients are not stationary. "
            f"Companion eigenvalue magnitudes: {np.round(np.abs(eigvals), 4)}"
        )

    innovation_bounds = centered_beta_support(
        alpha=beta_alpha,
        beta=beta_beta,
        support_width=support_width,
    )
    x = simulate_beta_ar(
        phi=phi,
        sample_size=n,
        intercept=intercept,
        support_width=support_width,
        alpha=beta_alpha,
        beta=beta_beta,
        burnin=burnin,
        rng=seed,
    )
    cloud2 = sliding_window_point_cloud(x, 2, newest_first=False)
    cloud3 = sliding_window_point_cloud(x, 3, newest_first=False)
    cloud2_plot = cloud2
    cloud3_plot = cloud3
    if len(cloud2) > max_points:
        idx2 = np.linspace(0, len(cloud2) - 1, max_points, dtype=int)
        cloud2_plot = cloud2[idx2]
    if len(cloud3) > max_points:
        idx3 = np.linspace(0, len(cloud3) - 1, max_points, dtype=int)
        cloud3_plot = cloud3[idx3]

    innovation_lower, innovation_upper = innovation_bounds
    if beta_alpha == 1.0 and beta_beta == 1.0:
        innovation_label = (
            rf"Uniform$[{innovation_lower:g},{innovation_upper:g}]$ innovations"
        )
    else:
        innovation_label = (
            rf"centered Beta$({beta_alpha:g},{beta_beta:g})$ innovations"
        )

    print(f"AR order p = {len(phi)}")
    print(f"Companion eigenvalues = {np.round(eigvals, 4)}")
    print(f"Sample mean = {x.mean():.4f}; sample SD = {x.std(ddof=1):.4f}")
    print(
        f"Centered Beta shapes = ({beta_alpha:g}, {beta_beta:g}); "
        f"innovation support = [{innovation_lower:.4f}, {innovation_upper:.4f}]"
    )
    if len(x) > max_points:
        print(
            "Plots are thinned for responsiveness; full arrays retain all "
            f"{len(x):,} observations."
        )

    plot_time_series(x, phi, innovation_label=innovation_label)
    plot_embedding_k2(
        cloud2_plot,
        phi,
        intercept,
        innovation_bounds=innovation_bounds,
    )
    plot_embedding_k3(
        cloud3_plot,
        phi,
        intercept,
        innovation_bounds=innovation_bounds,
    )
    return x, cloud2, cloud3


## Interactive controls

Enter coefficients in lag order, for example `0.7` for AR(1) or `0.6, -0.2` for AR(2). The Beta innovations are centered to have mean zero, and the support-width control fixes the total distance between their lower and upper bounds. With the default settings $\alpha=\beta=1$ and support width $2$, the innovations are exactly $\mathcal U[-1,1]$ as before. The sample-size field accepts values up to 1,000,000. For long series, plots are automatically thinned for responsiveness, while the complete arrays remain available as `x`, `cloud_k2`, and `cloud_k3`.


In [ ]:
phi_widget = widgets.Text(
    value="0.7",
    description="AR coefficients:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="360px"),
)
n_widget = widgets.BoundedIntText(
    value=1200,
    min=200,
    max=1_000_000,
    step=1000,
    description="Sample size:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="300px"),
)
support_width_widget = widgets.FloatSlider(
    value=2.0,
    min=0.2,
    max=6.0,
    step=0.2,
    description="Support width:",
    style={"description_width": "130px"},
)
alpha_widget = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description=r"Beta alpha:",
    style={"description_width": "130px"},
)
beta_widget = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description=r"Beta beta:",
    style={"description_width": "130px"},
)
intercept_widget = widgets.FloatSlider(
    value=0.0,
    min=-2.0,
    max=2.0,
    step=0.1,
    description="Intercept:",
    style={"description_width": "130px"},
)
seed_widget = widgets.IntText(
    value=7,
    description="Random seed:",
    style={"description_width": "130px"},
)
run_button = widgets.Button(
    description="Generate and plot",
    button_style="primary",
    icon="play",
)
output = widgets.Output()


def handle_run(_=None):
    with output:
        clear_output(wait=True)
        try:
            phi = parse_phi(phi_widget.value)
            global x, cloud_k2, cloud_k3
            x, cloud_k2, cloud_k3 = run_experiment(
                phi=phi,
                n=n_widget.value,
                intercept=intercept_widget.value,
                support_width=support_width_widget.value,
                beta_alpha=alpha_widget.value,
                beta_beta=beta_widget.value,
                burnin=max(500, 10 * len(phi)),
                seed=seed_widget.value,
            )
        except Exception as exc:
            print(f"Could not generate the requested process: {exc}")


run_button.on_click(handle_run)
controls = widgets.VBox([
    widgets.HTML("<b>AR process settings</b>"),
    phi_widget,
    widgets.HBox([n_widget, seed_widget]),
    widgets.HBox([alpha_widget, beta_widget]),
    widgets.HBox([support_width_widget, intercept_widget]),
    run_button,
])
display(controls, output)
handle_run()
